### Pre-processing 
This involves: 
* reading the Excel file
* dropping Nan rows (withdrawn, void, absent in work roles).
* reformat labels: some labels are formatted as 4,5. Others are labeled as 0.2. with/without spaces. Turn them into one-hot encodings. 
* drop knowledge unit column.  Knowledge units vs knowledge areas. The KAs may contain multiple knowledge units. Knowledge units are out of the scope of this project right now, but I should think of a way to number/label them. 
* convert to Pytorch tensor. Drop KD numbers column and headers before converting to Pytorch array. we don't need the KD numbers for training, just for interpreting the results. 

In [50]:
# import necessary packages
import pandas as pd

# read Excel file as Pandas df
path = "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/NICE Framework Components v2.0.0(2).xlsx"

des = pd.read_excel(path, sheet_name='v2.0.0 TKS Statements')
print(des.columns)
print(des)

Index(['TKS ID', 'TKS Description', 'New for v2.0.0'], dtype='object')
     TKS ID                                    TKS Description  New for v2.0.0
0     K0018                 Knowledge of encryption algorithms           False
1     K0055                       Knowledge of microprocessors           False
2     K0064  Knowledge of performance tuning tools and tech...           False
3     K0068  Knowledge of programming language structures a...           False
4     K0092      Knowledge of technology integration processes           False
...     ...                                                ...             ...
2106  T2047                                Inventory OT assets            True
2107  T2048  Recommend cybersecurity requirements for integ...            True
2108  T2049  Serve as OT engineering subject matter expert ...            True
2109  T2050  Serve as OT engineering subject matter expert ...            True
2110  T2051  Train cybersecurity defense technicians on OT .

### split in to train and test according to Harri's method 

In [51]:
import random 
numbers = []
for num in range(14, 633): 
    numbers.append(num)

random.seed(40)                 # set the seed for reproducibility
random.shuffle(numbers) 

first_50_numbers = numbers[:50] 
first_50_numbers.sort()
last_numbers = numbers[50:]
last_numbers.sort()
print(first_50_numbers)
print(last_numbers)

[19, 22, 24, 41, 84, 88, 91, 97, 110, 111, 135, 143, 151, 153, 154, 176, 179, 185, 192, 200, 207, 210, 221, 222, 278, 318, 325, 351, 354, 424, 461, 463, 473, 477, 506, 521, 523, 524, 527, 534, 544, 561, 571, 574, 575, 591, 592, 600, 606, 613]
[14, 15, 16, 17, 18, 20, 21, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 85, 86, 87, 89, 90, 92, 93, 94, 95, 96, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 144, 145, 146, 147, 148, 149, 150, 152, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 180, 181, 182, 183, 184, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 197, 198, 199,

In [52]:
df_test = des.apply(lambda x: x[first_50_numbers])
df_train = des.apply(lambda x: x[last_numbers])

In [53]:
df_train.index = [x for x in range(len(df_train))]

In [60]:
print(df_train['TKS Description'])
print(repr(df_train['TKS Description'][30]))

0                         Knowledge of data repositories
1               Knowledge of security awareness programs
2       Knowledge of code tailoring tools and techniques
3      Knowledge of the organizational cybersecurity ...
4      Knowledge of market research tools and techniques
                             ...                        
564                            Knowledge of OT protocols
565    Knowledge of process hazard analysis (PHA) ass...
566            Knowledge of system assets and boundaries
567             Skill in conducting information searches
568                      Skill in conducting test events
Name: TKS Description, Length: 569, dtype: object
'Knowledge of the Communications Security (COMSEC) Material Control System (CMCS)'


In [58]:
print(df['Statement Description'][0])
print(repr(df['Statement Description'][0]))

Knowledge of computer networking concepts and protocols, and network security methodologies
'Knowledge of computer networking concepts and protocols, and network security methodologies'


In [71]:
# find indexes of df that correspond to df_train 
l = []
for j in range(len(df_train)): 
    for idx,i in enumerate(range(len(df))): 
        score = sum(a == b for a,b in zip(df_train['TKS Description'][j], df['Statement Description'][i])) 
        if score >= round(min(0.9*len(df_train['TKS Description'][j]), 0.9*len(df['Statement Description'][i]))): 
           # print(idx)
            l.append(idx)

In [72]:
# check if all elements are unique 
if len(l) > len(set(l)): 
    print('no')

print(len(l))
print(len(set(l)))

no
56
49


In [42]:
for idx,i in enumerate(range(len(df_train))): 
    if df['Statement Description'][0] == df_train['TKS Description'][i] : 
        print(idx)

### Take the labels from the old mapping and keep all labeled instances. 

In [12]:
# import necessary packages
import pandas as pd

# read Excel file as Pandas df
path = "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/Mappings_between_CSEC2017_and_the_NICE_Framework.xlsx"

df = pd.read_excel(path, sheet_name='Mapping and KD-Weight')
print(df.columns)
print(df.index)

# drop all Nan rows (withdrawn, void, absent in work roles). 
df.dropna(axis=0, inplace=True) #0 = rows 
#print(df)

Index(['knowledge descriptions(NICE Framework)', 'Knowledge Units (CSEC2017)',
       ' Knowledge areas (CSEC2017)'],
      dtype='object')
RangeIndex(start=0, stop=630, step=1)


In [13]:
print(df)

    knowledge descriptions(NICE Framework)  \
0                                    K0001   
1                                    K0002   
2                                    K0003   
3                                    K0004   
4                                    K0005   
..                                     ...   
613                                  K0614   
614                                  K0615   
621                                  K0622   
623                                  K0624   
627                                  K0628   

                            Knowledge Units (CSEC2017)  \
0                                 Network Architecture   
1                                      Risk Management   
2       Cyber Law, Cyber Ethics, Cyber Policy, Privacy   
3     Fundamental Principles, System Thinking, Privacy   
4    Cyberspace Practice, Component Design, Network...   
..                                                 ...   
613  Communication and Networking, Networ

In [14]:
# some labels are formatted as 4,5. Others are labeled as 0.2. with/without spaces. 
# reformat so they are all the same format. 
def df_to_onehot(df, column=' Knowledge areas (CSEC2017)'): 
    # find all unique labels
    all_labels = set(label for row in df[column] for label in row.split(','))
    all_labels = list(all_labels)
    all_labels.sort()
    # 
    print(all_labels)
    # initialize one-hot encoding coluns with 0. 
    for label in all_labels: 
        df[label] = 0 

    # set 1 for all indicated labels in the specified column. 
    for index, row in df.iterrows(): 
        labels = row[column].split(',')
        for label in labels: 
            df.at[index,label] = 1 

def format_knowledge_areas(value): 
    # turn value into string for easier processing 
    value = str(value)
    # replace periods with commas 
    value = value.replace('.', ',') 
    # remove any extra spaces 
    value = value.replace(' ','')
    return value 
df[' Knowledge areas (CSEC2017)'] = df[' Knowledge areas (CSEC2017)'].apply(format_knowledge_areas)
df_to_onehot(df)
print(df)

['0', '1', '2', '3', '4', '5', '6', '7', '8']
    knowledge descriptions(NICE Framework)  \
0                                    K0001   
1                                    K0002   
2                                    K0003   
3                                    K0004   
4                                    K0005   
..                                     ...   
613                                  K0614   
614                                  K0615   
621                                  K0622   
623                                  K0624   
627                                  K0628   

                            Knowledge Units (CSEC2017)  \
0                                 Network Architecture   
1                                      Risk Management   
2       Cyber Law, Cyber Ethics, Cyber Policy, Privacy   
3     Fundamental Principles, System Thinking, Privacy   
4    Cyberspace Practice, Component Design, Network...   
..                                                 ..

In [15]:
# extract KD index column for result evaluation. 
KD_index = df['knowledge descriptions(NICE Framework)']
print(KD_index)
# drop KD index, KU and original KA columns. 
df = df.drop(columns=[' Knowledge areas (CSEC2017)','Knowledge Units (CSEC2017)'])

0      K0001
1      K0002
2      K0003
3      K0004
4      K0005
       ...  
613    K0614
614    K0615
621    K0622
623    K0624
627    K0628
Name: knowledge descriptions(NICE Framework), Length: 576, dtype: object


In [16]:
print(df)

    knowledge descriptions(NICE Framework)  0  1  2  3  4  5  6  7  8
0                                    K0001  0  0  0  0  1  0  0  0  0
1                                    K0002  0  0  0  0  0  0  0  1  0
2                                    K0003  0  0  0  0  0  0  0  0  1
3                                    K0004  0  0  1  0  0  1  0  0  1
4                                    K0005  1  0  0  1  1  1  0  0  0
..                                     ... .. .. .. .. .. .. .. .. ..
613                                  K0614  1  0  0  0  1  0  0  0  0
614                                  K0615  0  0  0  0  0  0  1  0  0
621                                  K0622  0  0  0  0  0  0  0  1  0
623                                  K0624  1  0  1  0  0  0  0  0  0
627                                  K0628  1  0  0  0  0  0  0  0  0

[576 rows x 10 columns]


In [17]:
# Add the actual content/generate another dataframe with the content. 
# read Excel file as Pandas df
path = "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/NICE Framework Components Mapping - 2017 to v1.0.0 March 2024.xlsx"

des = pd.read_excel(path, sheet_name='TKS Statements Mapping')
print(des.columns)
print(des.index)


Index(['KSAT ID', 'Statement Description', 'New', 'Withdrawn',
       'Replacement Statements (if applicable)'],
      dtype='object')
RangeIndex(start=0, stop=4344, step=1)


In [18]:
#rows_k = des[des.apply(lambda row: row.astype(str).str.startswith('K').any(), axis=1)]
rows_k = des['KSAT ID'].str.startswith('K', na=False)
#rows_k.dropna(axis=0, inplace=True)
kds = des[rows_k]
print(rows_k[:])
print(rows_k[1270])
print(kds)
# keep only the KSAT ID column and the statement descriptoin. 


0       False
1       False
2       False
3       False
4       False
        ...  
4339    False
4340    False
4341    False
4342    False
4343    False
Name: KSAT ID, Length: 4344, dtype: bool
True
     KSAT ID                              Statement Description    New  \
177    K0001  Knowledge of computer networking concepts and ...  False   
178    K0002  Knowledge of risk management processes (e.g., ...  False   
179    K0003  Knowledge of laws, regulations, policies, and ...  False   
180    K0004  Knowledge of cybersecurity and privacy principles  False   
181    K0005     Knowledge of cyber threats and vulnerabilities  False   
...      ...                                                ...    ...   
1431   K1271  Knowledge of system alert policies and procedures   True   
1432   K1272                     Knowledge of system components   True   
1433   K1273  Knowledge of threat investigation policies and...   True   
1434   K1274  Knowledge of threat modeling tools and techniq

In [19]:
print(kds.columns)
kds = kds.drop(columns=['New', 'Withdrawn', 'Replacement Statements (if applicable)'])

Index(['KSAT ID', 'Statement Description', 'New', 'Withdrawn',
       'Replacement Statements (if applicable)'],
      dtype='object')


In [20]:
print(kds.columns)

Index(['KSAT ID', 'Statement Description'], dtype='object')


In [21]:
# index the descriptions. 
kds_old = kds[kds['KSAT ID'].isin(KD_index)]
print(kds_old)

    KSAT ID                              Statement Description
177   K0001  Knowledge of computer networking concepts and ...
178   K0002  Knowledge of risk management processes (e.g., ...
179   K0003  Knowledge of laws, regulations, policies, and ...
180   K0004  Knowledge of cybersecurity and privacy principles
181   K0005     Knowledge of cyber threats and vulnerabilities
..      ...                                                ...
790   K0614  Knowledge of wireless technologies (e.g., cell...
791   K0615  Knowledge of privacy disclosure statements bas...
798   K0622  Knowledge of controls related to the use, proc...
800   K0624  Knowledge of Application Security Risks (e.g. ...
804   K0628  Knowledge of cyber competitions as a way of de...

[576 rows x 2 columns]


In [22]:
df.rename(columns={"knowledge descriptions(NICE Framework)":"KSAT ID"}, inplace=True)
df = df.merge(kds_old, on="KSAT ID", how='left')
print(df.columns)
print(kds_old.columns)

Index(['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8',
       'Statement Description'],
      dtype='object')
Index(['KSAT ID', 'Statement Description'], dtype='object')


In [23]:
print(df)
print(df.columns)

    KSAT ID  0  1  2  3  4  5  6  7  8  \
0     K0001  0  0  0  0  1  0  0  0  0   
1     K0002  0  0  0  0  0  0  0  1  0   
2     K0003  0  0  0  0  0  0  0  0  1   
3     K0004  0  0  1  0  0  1  0  0  1   
4     K0005  1  0  0  1  1  1  0  0  0   
..      ... .. .. .. .. .. .. .. .. ..   
571   K0614  1  0  0  0  1  0  0  0  0   
572   K0615  0  0  0  0  0  0  1  0  0   
573   K0622  0  0  0  0  0  0  0  1  0   
574   K0624  1  0  1  0  0  0  0  0  0   
575   K0628  1  0  0  0  0  0  0  0  0   

                                 Statement Description  
0    Knowledge of computer networking concepts and ...  
1    Knowledge of risk management processes (e.g., ...  
2    Knowledge of laws, regulations, policies, and ...  
3    Knowledge of cybersecurity and privacy principles  
4       Knowledge of cyber threats and vulnerabilities  
..                                                 ...  
571  Knowledge of wireless technologies (e.g., cell...  
572  Knowledge of privacy disclosure st

In [24]:
# now save in the desired format (pytorch tensor?) for processing by huggingface
from datasets import Dataset 
dataset = Dataset.from_pandas(df)
print(dataset)

Dataset({
    features: ['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description'],
    num_rows: 576
})


In [104]:
# save to disk. 
dataset.save_to_disk("data/train.hf")

Saving the dataset (0/1 shards):   0%|          | 0/576 [00:00<?, ? examples/s]

In [5]:
# load from disk. 
from datasets import load_from_disk
ds = load_from_disk("data/train.hf")

In [6]:
print(ds)

Dataset({
    features: ['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description'],
    num_rows: 576
})
